# Global daily budget climatology

Day-of-year climatologies of the ACCESS-OM2 online mixed-layer temperature budget terms,
computed with `mhw3d.best_practice.compute_climatology`.

**Run this before NB01–NB03.** They open the file it writes.

## Why

The budget climatology these notebooks used up to now was a **monthly** mean
(`mlt_budget_stavg_daily_online_output336-365_monthly_mean.ncea.nc`), linearly interpolated
to daily inside each notebook. In seasonally ice-covered regions the seasonal cycle is
*truncated*: near-flat through the ice-covered months, then a short, sharp summer excursion.
Twelve monthly values linearly interpolated cannot represent the shoulders of that cycle, so
the interpolated climatology aliased them into the daily budget anomalies — in exactly the
regions this chapter is about.

Computing the climatology directly from the daily budget output removes that aliasing, and
uses the same day-of-year construction (11-day window, 31-day smoothing) as
`om2_025_MLT_clim.nc` / `om2_025_MLT_thresh.nc`, so budget anomalies and MHW detection rest
on the same baseline.

## Why global

The Arctic study needs the same climatology, and a day-of-year climatology is generally
better posed than a monthly one wherever a sharp or short seasonal feature matters. One
global file serves both hemispheres; regional notebooks slice it by latitude on load.

## How it scales

`compute_climatology` is lazy end to end — it never calls `.compute()`, `.load()` or
`.values` on the data, and its one internal rechunk collapses only the day-of-year axis
(~428 long), leaving the spatial chunking alone. So the whole globe runs as **one dask graph**
and `to_netcdf` streams it to disk. Nothing here holds a global field in memory, and there is
no latitude banding.

What *does* matter is the input chunking, because `rolling(time=2*windowHalfWidth+1)` builds
a windowed view and dask rechunks to keep that intermediate bounded. Measured by building the
graph on the true global shape (30 yr daily × 1080 × 1440):

| `YT_CHUNK` | input chunk | rolling chunk | tasks/term | × 8 terms |
|---:|---:|---:|---:|---:|
| 1080 | 2271 MB | 143.9 MB | 30,263 | 242,104 |
| **540** | **1135 MB** | **71.9 MB** | **60,491** | **483,928** |
| 360 | 757 MB | 48.0 MB | 90,719 | 725,752 |
| 216 | 454 MB | 28.8 MB | 151,175 | 1,209,400 |

Coarse chunking gives the smallest graph but forces multi-GB reads per task; fine chunking
keeps per-task memory small but pushes the graph past a million tasks, where the distributed
scheduler starts to struggle. The default is the compromise. Section 4 reports where you
actually landed *before* anything is computed.

For reference: splitting the globe into five 216-row latitude bands comes to 151,315 tasks in
total — the same work and the same intermediate size as one pass, at the cost of five sets of
file opens and a merge step. Banding buys nothing here.

## Baseline period

Outputs 336–365 = 1989–2018, matching the climatology period of Holmes & Malan (2026) and of
the existing monthly budget climatology file. (output305 = 1958, so output *NNN* = 1958 + *NNN* − 305.)

In [ ]:
import os
import time

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from mhw3d import best_practice

%matplotlib inline

In [ ]:
from dask.distributed import Client

# processes=True + threads_per_worker=1 avoids netCDF4/HDF5 thread-safety races
# when many workers read chunks from the same open_mfdataset files concurrently.
# Do NOT fall back to dask's default threaded scheduler here: concurrent reads
# raise "NetCDF: HDF error" or corrupt the heap outright.
client = Client(processes=True, threads_per_worker=1, n_workers=12)
client

## 1. Configuration

In [ ]:
# ── Data ──────────────────────────────────────────────────────────────────────
base = '/g/data/av17/access-nri/OM2/025deg_jra55_iaf_cycle6_online_mlt/'
BUDGET_SUBDIR = 'post_processed_diags/mlt_budget_online_stavg/'
MONTHLY_CLIM_FILE = (base + BUDGET_SUBDIR +
                     'mlt_budget_stavg_daily_online_output336-365_monthly_mean.ncea.nc')

OUTPUT_DIR = '/scratch/m35/nm5072/Polar_MHWs/'
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUT_FILE = OUTPUT_DIR + 'mlt_budget_clim_daily_336-365_global.nc'

# Climatology baseline: outputs 336-365 = 1989-2018
START_OUTPUT = 336
END_OUTPUT   = 365
outputs      = list(range(START_OUTPUT, END_OUTPUT + 1))

def output_to_year(o):
    return 1958 + o - 305

# ── Domain ────────────────────────────────────────────────────────────────────
# None = full global grid. Set both to restrict (e.g. -80.0 / -60.0 for the
# Antarctic strip alone).
LAT_MIN = None
LAT_MAX = None

# TEST_MODE = True runs 60 latitude rows x 120 longitudes end to end in minutes.
# Use it to check the whole notebook before committing a full PBS job.
TEST_MODE = False

# ── Budget terms ──────────────────────────────────────────────────────────────
VARS_RAW = ['mlt_tendency', 'advection', 'vert_mixing',
            'entrainment', 'surface_flux', 'sw_pen', 'residual']

# 'surf_to_ML' is derived below and is the term NB01-NB03 use for the surface
# flux; 'surface_flux' and 'sw_pen' are kept separately so the shortwave
# contribution can be diagnosed.
CLIM_TERMS = ['mlt_tendency', 'surf_to_ML', 'advection', 'vert_mixing',
              'entrainment', 'residual', 'surface_flux', 'sw_pen']

# ── mhw3d climatology parameters ──────────────────────────────────────────────
# WINDOW_HALF_WIDTH=5 pools +/-5 calendar days before the day-of-year mean, i.e.
# the 11-day window of Hobday et al. (2016); SMOOTH_MEAN_WIDTH=31 is the 31-day
# smoothing of the resulting seasonal cycle. Both match the construction of
# om2_025_MLT_clim.nc / om2_025_MLT_thresh.nc, so budget anomalies and MHW
# detection share a baseline. Lower SMOOTH_MEAN_WIDTH if the 31-day smoother is
# found to still over-smooth the shoulders of the truncated polar cycle -- but
# then the MLT climatology should be recomputed to match.
WINDOW_HALF_WIDTH = 5
SMOOTH_MEAN       = True
SMOOTH_MEAN_WIDTH = 31

# ── Chunking ──────────────────────────────────────────────────────────────────
# See the table in the header. xt_ocean is left whole so reads stay contiguous in
# longitude and no chunk boundary lands on the grid seam.
TIME_CHUNK = 365
YT_CHUNK   = 540

# NetCDF chunking of the output, sized for latitude-slice reads (~7 MB/chunk).
OUT_CHUNKSIZES = (61, 120, 240)   # (dayofyear, yt_ocean, xt_ocean)
COMPLEVEL      = 4                # 0 disables compression

SEC_PER_DAY = 86400.0

print(f'Baseline : outputs {START_OUTPUT}-{END_OUTPUT} '
      f'({output_to_year(START_OUTPUT)}-{output_to_year(END_OUTPUT)})')
print(f'Output   : {OUT_FILE}')
print(f'Terms    : {CLIM_TERMS}')

## 2. Running as a PBS batch job on Gadi

Section 5 is the expensive step. Submit headless so it survives terminal disconnection:

```bash
cat > run_budget_clim.pbs << 'EOF'
#!/bin/bash
#PBS -N mlt_budget_clim
#PBS -q normal
#PBS -l walltime=10:00:00      # a GUESS — see below
#PBS -l mem=190GB
#PBS -l ncpus=12
#PBS -l storage=gdata/av17+gdata/xp65+gdata/m35+scratch/m35
#PBS -l wd
#PBS -j oe

set -euo pipefail
PROJECT=m35

module use /g/data/xp65/public/modules
module load conda/analysis3
source /g/data/${PROJECT}/${USER}/venvs/mhw3d/bin/activate

python3 -c "import mhw3d; print('mhw3d:', mhw3d.__file__)"
python3 -c "import flox; print('flox:', flox.__version__)" \
    || echo "WARNING: flox missing — groupby('time.dayofyear') falls back to a slow path"

jupyter nbconvert --to notebook --execute --inplace \
    --ExecutePreprocessor.timeout=36000 \
    notebooks/00_budget_clim_daily.ipynb
EOF
qsub run_budget_clim.pbs
```

**The walltime and memory above have not been measured against the real files.** Two cheap
checks first:

1. Run sections 1–4 interactively (ARE, or a login node). Section 4 builds the graph and
   reports its size without computing anything — seconds.
2. Set `TEST_MODE = True` and run the whole notebook. Scale its runtime by the ratio of grid
   points to estimate the real walltime, then flip back to `False`.

**There is no checkpoint** — a single streaming write either finishes or it doesn't. If a run
dies, narrow `CLIM_TERMS` and do the terms in groups; they are separate variables in the
source files, so a subset costs little extra I/O.

## 3. Load the daily budget files (lazy)

In [ ]:
def budget_path(output):
    return f'{base}{BUDGET_SUBDIR}mlt_budget_stavg_daily_online_output{output:03d}.nc'


# Resolve which outputs actually exist — a short baseline must not pass silently.
budget_files, outputs_found, missing = [], [], []
for o in outputs:
    p = budget_path(o)
    (budget_files.append(p), outputs_found.append(o)) if os.path.exists(p) else missing.append(o)

if missing:
    print(f'WARNING: {len(missing)} of {len(outputs)} budget files missing — '
          f'outputs {missing} (years {[output_to_year(o) for o in missing]}).')
    print('The climatology will use the available years only; this is recorded in '
          'baseline_years / outputs_used in the output attributes.')
if not budget_files:
    raise FileNotFoundError(f'No daily budget files found under {base}{BUDGET_SUBDIR}')

years_found = [output_to_year(o) for o in outputs_found]
print(f'Budget files: {len(budget_files)} '
      f'(outputs {outputs_found[0]}-{outputs_found[-1]}, '
      f'years {years_found[0]}-{years_found[-1]})')

In [ ]:
ds = xr.open_mfdataset(
    budget_files,
    decode_times=True,
    chunks={'time': TIME_CHUNK, 'yt_ocean': YT_CHUNK, 'xt_ocean': -1},
    combine='nested', concat_dim='time',
    parallel=True, decode_timedelta=False,
)[VARS_RAW]

if LAT_MIN is not None or LAT_MAX is not None:
    ds = ds.sel(yt_ocean=slice(LAT_MIN, LAT_MAX))
if TEST_MODE:
    ds = ds.isel(yt_ocean=slice(0, 60), xt_ocean=slice(0, 120))
    print('TEST_MODE: 60 latitude rows, 120 longitudes.')

ds['surf_to_ML'] = ds['surface_flux'] + ds['sw_pen']

doy = ds.time.dt.dayofyear
print(f'Domain    : {dict(ds.sizes)}')
print(f'yt_ocean  : {float(ds.yt_ocean.min()):.2f} to {float(ds.yt_ocean.max()):.2f} '
      f'(nominal deg — tripolar north of ~65N, see the note in section 6)')
print(f'time span : {str(ds.time.values[0])[:10]} to {str(ds.time.values[-1])[:10]}  '
      f'({ds.sizes["time"]} days, {ds.sizes["time"] / len(budget_files):.1f} days/file)')
print(f'dayofyear : {int(doy.min())}-{int(doy.max())} '
      f'(366 ⇒ the calendar includes leap days)')

## 4. Build the climatology graph

Nothing is computed here. The printout is the cheap sanity check before committing a job:
the graph should be a few hundred thousand tasks, the rolling intermediate tens of MB, and
`still lazy` must be `True`. If the numbers look wrong, change `YT_CHUNK` in section 1 and
re-run this cell — no data is read either way.

In [ ]:
def chunk_mb(chunks):
    return float(np.prod([max(d) for d in chunks])) * 4 / 1e6


t0 = time.time()
clim = xr.Dataset({
    term: best_practice.compute_climatology(
        ds[term],
        smoothMean=SMOOTH_MEAN,
        smoothMeanWidth=SMOOTH_MEAN_WIDTH,
        windowHalfWidth=WINDOW_HALF_WIDTH,
        baseline_period=None,   # files are already restricted to the baseline
    )
    for term in CLIM_TERMS
})

roll = ds[CLIM_TERMS[0]].rolling(
    time=2 * WINDOW_HALF_WIDTH + 1, center=True, min_periods=1).mean()
n_tasks = sum(len(clim[t].data.__dask_graph__()) for t in CLIM_TERMS)

print(f'Graph: {n_tasks:,} tasks for {len(CLIM_TERMS)} terms '
      f'(built in {time.time() - t0:.1f} s, nothing computed)')
print(f'  input chunk   : {chunk_mb(ds[CLIM_TERMS[0]].chunks):8.1f} MB')
print(f'  rolling chunk : {chunk_mb(roll.chunks):8.1f} MB   ← the memory-critical intermediate')
print(f'  output chunk  : {chunk_mb(clim[CLIM_TERMS[0]].chunks):8.2f} MB')
print(f'  still lazy    : {all(hasattr(clim[t].data, "chunks") for t in CLIM_TERMS)}')

## 5. Write to NetCDF

`to_netcdf` streams the graph chunk by chunk — the global field is never resident.

The one thing worth getting right is chunk alignment. The rolling stage leaves the output in
many small spatial chunks; writing those into larger on-disk NetCDF chunks makes HDF5
read-modify-write the same chunk repeatedly. Rechunking first to match the on-disk chunking
took a test case from 4.0 min to 1.1 min, and produced a *smaller* file (aligned writes
compress better).

In [ ]:
for term in CLIM_TERMS:
    clim[term].attrs.update({
        'long_name': f'Day-of-year climatology of {term}',
        'units': 'degC s-1',
    })

clim.attrs.update({
    'description': ('Global daily (day-of-year) climatology of ACCESS-OM2 online '
                    'mixed-layer temperature budget terms'),
    'method': ('mhw3d.best_practice.compute_climatology '
               '(github.com/ocean-mhw/mhw3d-detection)'),
    'windowHalfWidth': WINDOW_HALF_WIDTH,
    'smoothMean': str(SMOOTH_MEAN),
    'smoothMeanWidth': SMOOTH_MEAN_WIDTH,
    'baseline_outputs': f'{outputs_found[0]}-{outputs_found[-1]}',
    'baseline_years': f'{years_found[0]}-{years_found[-1]}',
    'n_years_used': len(outputs_found),
    'outputs_used': ','.join(str(o) for o in outputs_found),
    'terms': ','.join(CLIM_TERMS),
    'model': 'ACCESS-OM2 0.25 deg JRA55 IAF cycle 6, online MLT budget',
    'source_files': BUDGET_SUBDIR + 'mlt_budget_stavg_daily_online_output{NNN}.nc',
    'grid_note': ('Tripolar north of ~65N: yt_ocean is nominal there. Use '
                  'geolat_t/geolon_t for Arctic region selection.'),
    'note': ('Replaces the monthly-mean budget climatology '
             '(output336-365_monthly_mean.ncea.nc), whose monthly-to-daily '
             'interpolation aliased the truncated seasonal cycle in seasonally '
             'ice-covered regions.'),
    'created_by': 'notebooks/00_budget_clim_daily.ipynb',
})

# Align the dask chunks with the on-disk chunking before writing.
clim_w = clim.chunk({'dayofyear': -1,
                     'yt_ocean': OUT_CHUNKSIZES[1],
                     'xt_ocean': OUT_CHUNKSIZES[2]})
print(f'write chunk: {chunk_mb(clim_w[CLIM_TERMS[0]].chunks):.1f} MB')

encoding = {}
for v in clim_w.data_vars:
    enc = {'dtype': 'float32'}
    if COMPLEVEL > 0:
        enc.update({'zlib': True, 'complevel': COMPLEVEL,
                    'chunksizes': tuple(min(c, clim_w.sizes[d]) for c, d in
                                        zip(OUT_CHUNKSIZES,
                                            ('dayofyear', 'yt_ocean', 'xt_ocean')))})
    encoding[v] = enc

t0 = time.time()
tmp = OUT_FILE + '.tmp'          # write-then-rename, so a killed job leaves no
clim_w.to_netcdf(tmp, encoding=encoding)   # half-written file that looks complete
os.replace(tmp, OUT_FILE)

print(f'Saved → {OUT_FILE}')
print(f'  {os.path.getsize(OUT_FILE) / 1e9:.2f} GB in {(time.time() - t0) / 60:.1f} min')

## 6. Verify the output

Reopen the file from disk and check it, then compare the new climatology against the one it
replaces at an ice-affected point.

The raw day-of-year mean is a *shape* reference, not a target: it is computed from whatever
years are loaded here with no smoothing at all, while both climatologies use the full
1989–2018 baseline. What matters is whether the shoulders and the timing of the summer
excursion in the new curve track the raw one more closely than the old curve does.

In [ ]:
chk = xr.open_dataset(OUT_FILE)
print(chk)

assert set(CLIM_TERMS) <= set(chk.data_vars), 'missing terms in output'
assert bool((np.diff(chk.yt_ocean.values) > 0).all()), 'yt_ocean not monotonic'
print(f'\ndayofyear   : {int(chk.dayofyear.min())}-{int(chk.dayofyear.max())}')
print(f'baseline    : {chk.attrs["baseline_years"]} '
      f'(outputs {chk.attrs["baseline_outputs"]}, {chk.attrs["n_years_used"]} years)')
print(f'smoothing   : windowHalfWidth={chk.attrs["windowHalfWidth"]}, '
      f'smoothMeanWidth={chk.attrs["smoothMeanWidth"]}')
for v in CLIM_TERMS:
    frac = float(np.isfinite(chk[v].values).mean())
    print(f'  {v:14s} finite fraction {frac:.3f}   '
          f'range {float(np.nanmin(chk[v])) * SEC_PER_DAY:+.4f} to '
          f'{float(np.nanmax(chk[v])) * SEC_PER_DAY:+.4f} °C day⁻¹')

In [ ]:
# ── Old (monthly-interpolated) vs new (daily) at one ice-affected point ───────
VALIDATION_TERM = 'surf_to_ML'
VAL_LAT, VAL_LON = -65.0, -71.0        # Drake Passage / Peninsula margin

sel = dict(yt_ocean=VAL_LAT, xt_ocean=VAL_LON, method='nearest')
pt = chk[VALIDATION_TERM].sel(**sel)
print(f'Validation point: yt_ocean={float(pt.yt_ocean):.2f}, '
      f'xt_ocean={float(pt.xt_ocean):.2f}')

# --- old method: 12 monthly means -> daily interpolation -> 31-day smooth ---
mclim = xr.open_dataset(MONTHLY_CLIM_FILE)
mclim['surf_to_ML'] = mclim['surface_flux'] + mclim['sw_pen']
mpt = mclim[VALIDATION_TERM].sel(**sel).compute()

by_month = mpt.groupby('time.month').mean('time').assign_coords(month=np.arange(1, 13))
padded = xr.concat([by_month.sel(month=12).expand_dims(month=[0]),
                    by_month,
                    by_month.sel(month=1).expand_dims(month=[13])], dim='month')
old_curve = (padded.interp(month=np.linspace(1, 12, 365))
             .rolling(month=31, center=True, min_periods=1).mean().values) * SEC_PER_DAY

new_curve = pt.sel(dayofyear=slice(1, 365)).values * SEC_PER_DAY

# --- raw day-of-year mean of the loaded daily series, unsmoothed ---
raw = (ds[VALIDATION_TERM].sel(**sel)
       .groupby('time.dayofyear').mean().compute()) * SEC_PER_DAY

doy_axis = np.arange(1, 366)
fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True,
                         gridspec_kw={'height_ratios': [2, 1]})

axes[0].plot(raw.dayofyear, raw.values, color='0.6', lw=1,
             label=f'raw DOY mean, {years_found[0]}–{years_found[-1]} (unsmoothed)')
axes[0].plot(doy_axis, old_curve, color='tab:orange', lw=2,
             label='old: monthly → daily interp + 31-day smooth')
axes[0].plot(doy_axis, new_curve, color='tab:blue', lw=2,
             label='new: mhw3d daily climatology')
axes[0].axhline(0, color='k', lw=0.5)
axes[0].set_ylabel('°C day⁻¹')
axes[0].legend(fontsize=9)
axes[0].set_title(f'{VALIDATION_TERM} climatology — '
                  f'{float(pt.yt_ocean):.2f}°, {float(pt.xt_ocean):.2f}°', fontsize=12)

axes[1].plot(doy_axis, new_curve - old_curve, color='tab:red', lw=1.5)
axes[1].axhline(0, color='k', lw=0.5)
axes[1].set_ylabel('new − old\n(°C day⁻¹)')
axes[1].set_xlabel('Day of year')
axes[1].set_xlim(1, 365)
plt.tight_layout()

rms = float(np.sqrt(np.nanmean((new_curve - old_curve) ** 2)))
rng = float(np.nanmax(new_curve) - np.nanmin(new_curve))
print(f'RMS(new − old)                       : {rms:.4f} °C day⁻¹')
print(f'Seasonal range of new                : {rng:.4f} °C day⁻¹')
print(f'RMS difference as % of seasonal range: {100 * rms / rng:.1f}%')

**Grid note.** ACCESS-OM2 is tripolar north of ~65°N, where `yt_ocean` is a nominal
coordinate rather than true latitude. This does not affect the climatology — it is a
pointwise day-of-year reduction — but Arctic *region selection* in the analysis notebooks
should use the 2D `geolat_t`/`geolon_t` fields, not `yt_ocean`.

In [ ]:
client.close()